## Installations

In [1]:
!pip install -q gradio langchain-community langchain-core faiss-cpu sentence-transformers pymupdf python-pptx transformers accelerate bitsandbytes reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 9.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.

## Imports

In [2]:
import re, io, time, random, difflib, urllib.parse, urllib.request
import xml.etree.ElementTree as ET
from collections import Counter
from datetime import date, datetime, timedelta
from math import ceil
from pathlib import Path
from xml.sax.saxutils import escape

import torch
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gradio as gr
import pymupdf
from pptx import Presentation

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

/tmp/ipykernel_3302/2807815316.py:20: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Model Loading

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=bnb_config, trust_remote_code=True
)
print("Model loaded successfully!")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully!


## Generate Function

In [4]:
def generate(prompt, max_new_tokens=300, sample=False):

    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)

    if sample:
        kwargs.update(do_sample=True, temperature=0.7, top_p=0.9)
    else:
        kwargs.update(do_sample=False)

    with torch.no_grad():
        outputs = model.generate(**inputs, **kwargs)

    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

## Global State

In [5]:
LETTERS = ["A", "B", "C", "D"]
WEAK_THRESHOLD = 0.5
COVERAGE_THRESHOLD = 0.7
MAX_RETRIES = 2
MAX_ROUNDS = 3
MAX_BATCH = 8

STOP_WORDS = {"The", "This", "That", "These", "Those", "And", "For", "With", "From", "Page", "Then",
              "When", "What", "Which", "Each", "Some", "Such", "Definition", "Example", "Examples",
              "Notes", "Outline", "Introduction", "Overview", "Summary", "Conclusion", "Agenda",
              "Questions", "Thanks"}

state = {}

def reset_practice():
    state.update(attempts=[], wrong=[], round=0, batch=[], asked=set(), checked=False, pool=[], plan=[])

reset_practice()

## Load Lecture (Parser)

In [6]:
def guess_title(text):
    for line in text.splitlines():
        if line.strip():
            return line.strip()[:100]
    return "(untitled)"


def load_lecture(path):

    ext = Path(path).suffix.lower()
    items = []

    if ext == ".pdf":
        doc = pymupdf.open(path)
        items = [(i, p.get_text().strip()) for i, p in enumerate(doc, start=1)]

    elif ext == ".pptx":
        prs = Presentation(path)
        for i, slide in enumerate(prs.slides, start=1):
            parts = [s.text_frame.text for s in slide.shapes if s.has_text_frame and s.text_frame.text.strip()]
            if slide.has_notes_slide:
                parts.append("Notes: " + slide.notes_slide.notes_text_frame.text)
            items.append((i, "\n".join(parts).strip()))
    else:
        raise ValueError("Unsupported file type: " + ext)

    pages, skipped = [], []

    for i, text in items:
        if len(text) < 30:
            skipped.append(i)
            continue
        pages.append(Document(page_content=text,
                              metadata={"page": i, "title": guess_title(text), "source": Path(path).name}))

    return pages, skipped

## Retriever Agent

In [7]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


def retriever_agent(query, k=3):

    results = state["db"].similarity_search(query, k=k)

    return "\n\n".join(f"[Page {d.metadata['page']}] {d.page_content}" for d in results)

/tmp/ipykernel_3302/2984461034.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Text Utilities

In [8]:
def extract_terms(text):
    terms = set(re.findall(r"\b[A-Z][A-Za-z0-9\-]{2,}\b", text))
    return {t for t in terms if t not in STOP_WORDS}


def stem(word):
    w = word.lower()
    for suf in ("ing", "ies", "es", "s"):
        if w.endswith(suf) and len(w) - len(suf) >= 3:
            return w[:-len(suf)]
    return w


def get_keywords(pages, top_n=12):
    counter, seen, out = Counter(), set(), []
    for p in pages:
        counter.update(extract_terms(p.page_content))
    for term, _ in counter.most_common():
        if stem(term) in seen:
            continue
        seen.add(stem(term))
        out.append(term)
        if len(out) == top_n:
            break
    return out


def clean_explanation(text, max_sentences=2):
    if "Not covered" in text:
        return "Not covered in the lecture."
    sents = re.split(r"(?<=[.!?])\s+", text.replace("\n", " ").strip())
    return " ".join(sents[:max_sentences]).strip()


def support_score(text, context):
    words = [w for w in re.findall(r"[a-z]+", text.lower()) if len(w) >= 3]
    if not words:
        return 0.0
    ctx = {stem(w) for w in re.findall(r"[a-z]+", context.lower())}
    return sum(1 for w in words if stem(w) in ctx) / len(words)


def parse_keywords(text):
    return [k.strip() for k in text.split(",") if k.strip()]

## Summarizer + Validator + Supervisor Agents

In [9]:
def make_sections(pages, max_chars=800):
    sections, cur, size = [], [], 0
    for p in pages:
        if cur and size + len(p.page_content) > max_chars:
            sections.append(cur); cur, size = [], 0
        cur.append(p); size += len(p.page_content)
    if cur:
        sections.append(cur)
    return sections


def summarizer_agent(section, missing_terms=None):

    text = "\n\n".join(f"[Page {d.metadata['page']}]\n{d.page_content}" for d in section)
    extra = ("Make sure these terms appear in your summary: " + ", ".join(missing_terms[:8]) + "\n") if missing_terms else ""

    prompt = f"""
You are a teaching assistant. Summarize the lecture pages below for exam preparation.

Rules:
- Write a section for EVERY page, with the heading "Page N" followed by 2-4 bullet points.
- Keep all definitions, formulas, numbers, and named concepts.
- Use only the text provided. Do not add outside facts.
{extra}
Lecture pages:
{text}
"""
    return generate(prompt, max_new_tokens=500)


def validator_agent(section, summary):

    terms = extract_terms(" ".join(d.page_content for d in section))

    if not summary:
        return 0.0, sorted(terms)

    summary_stems = {stem(w) for w in re.findall(r"[a-z0-9]+", summary.lower())}
    missing = []

    for t in terms:
        parts = [stem(p) for p in re.findall(r"[A-Za-z0-9]+", t)]
        if not all(p in summary_stems or difflib.get_close_matches(p, summary_stems, n=1, cutoff=0.8) for p in parts):
            missing.append(t)

    return 1 - len(missing) / max(len(terms), 1), sorted(missing)


def supervisor_agent(pages, progress=None):

    sections = make_sections(pages)
    results = []

    for i, section in enumerate(sections):

        first, last = section[0].metadata["page"], section[-1].metadata["page"]

        if progress:
            progress((i, len(sections)), desc=f"Summarizing pages {first}-{last}")

        best = summarizer_agent(section)
        cov, miss = validator_agent(section, best)
        retries = 0

        while cov < COVERAGE_THRESHOLD and retries < MAX_RETRIES:
            new = summarizer_agent(section, miss)
            c2, m2 = validator_agent(section, new)
            if c2 > cov:
                best, cov, miss = new, c2, m2
            retries += 1

        results.append({"pages": (first, last), "summary": best, "coverage": cov, "missing": miss})

    return results


def build_final_summary(results):
    return "\n\n---\n\n".join(f"## Pages {r['pages'][0]}-{r['pages'][1]}\n\n{r['summary']}" for r in results)

## Keyword Explainer Agent

In [10]:
def keyword_explainer_agent(term):

    context = retriever_agent(term, k=2)

    prompt = f"""
You are a teaching assistant.
Explain the term "{term}" in ONE or TWO short sentences, using ONLY the words and ideas in the context below.
If the context does not explain the term, answer exactly: Not covered in the lecture.

Context:
{context}
"""
    explanation = clean_explanation(generate(prompt, max_new_tokens=120))

    return explanation, support_score(explanation, context)


def build_glossary(terms, progress=None):

    entries = []

    for i, term in enumerate(terms):

        if progress:
            progress((i, len(terms)), desc=f"Explaining: {term}")

        explanation, score = keyword_explainer_agent(term)

        flag = ""
        if not explanation.startswith("Not covered") and score < 0.5:
            flag = "  ⚠ low grounding: may include outside knowledge"

        entries.append(f"**{term}**: {explanation}{flag}")

    return "\n\n".join(entries)

## Comparison Agent

In [11]:
def attribute_agent(concept, criterion):

    context = retriever_agent(concept, k=2)

    prompt = f"""
Using ONLY the context, answer in one short sentence.

Concept: {concept}
Question: What is its {criterion}?
If the context does not say, write: Not covered.

Context:
{context}
"""
    return clean_explanation(generate(prompt, max_new_tokens=100), max_sentences=1).replace("|", "/")


def comparison_agent(concepts, criteria):

    header = "| Criteria | " + " | ".join(concepts) + " |"
    divider = "|---" * (len(concepts) + 1) + "|"
    rows = [f"| {c} | " + " | ".join(attribute_agent(k, c) for k in concepts) + " |" for c in criteria]

    return "\n".join([header, divider] + rows)

## Resource Finder Agent (arXiv API)

In [12]:
def resource_finder_agent(term, max_results=2):

    query = urllib.parse.quote(f'all:"{term}" AND all:agents')
    url = f"https://export.arxiv.org/api/query?search_query={query}&max_results={max_results}"

    try:
        with urllib.request.urlopen(url, timeout=20) as response:
            root = ET.fromstring(response.read())

        ns = {"a": "http://www.w3.org/2005/Atom"}

        return [{"title": " ".join(e.find("a:title", ns).text.split()), "link": e.find("a:id", ns).text.strip()}
                for e in root.findall("a:entry", ns)]

    except Exception as e:
        print("Resource Finder failed for", term, ":", e)
        return []

## Question Generator Agent

In [13]:
def build_question(qtype, topic, page, question, options, answer_idx, explanation):
    return {"id": None, "type": qtype, "topic": topic, "page": page, "question": question,
            "options": options, "answer": answer_idx, "explanation": explanation}


def page_topic(page, keywords):
    stems_in_page = {stem(w) for w in re.findall(r"[A-Za-z0-9]+", page.page_content)}
    for k in keywords:
        if all(stem(p) in stems_in_page for p in re.findall(r"[A-Za-z0-9]+", k)):
            return k
    return f"Page {page.metadata['page']}"


def cloze_questions(pages, keywords, per_term=2):

    rng, questions = random.Random(42), []

    for term in keywords:

        others = [k for k in keywords if k != term]
        if len(others) < 3:
            continue

        pattern = re.compile(r"\b" + re.escape(term) + r"\b", re.IGNORECASE)
        found = 0

        for p in pages:
            for line in re.split(r"(?<=[.!?])\s+|\n", p.page_content):
                line = line.strip()
                if 40 <= len(line) <= 220 and pattern.search(line):
                    options = rng.sample(others, 3) + [term]
                    rng.shuffle(options)
                    questions.append(build_question("cloze", term, p.metadata["page"],
                                                    "Fill in the blank: " + pattern.sub("_____", line, count=1),
                                                    options, options.index(term),
                                                    f"From page {p.metadata['page']}: {line}"))
                    found += 1
                    break
            if found >= per_term:
                break

    return questions


def parse_mcq(raw):

    q = re.search(r"Question:\s*(.+)", raw)
    ans = re.search(r"Answer:\s*\(?([A-D])", raw)
    opts = re.findall(r"^\s*([A-D])[\)\.]\s*(.+)$", raw, re.MULTILINE)

    if not q or not ans or len(opts) != 4 or [l for l, _ in opts] != LETTERS:
        return None

    texts = [t.strip() for _, t in opts]

    if len({t.lower() for t in texts}) < 4:
        return None

    return q.group(1).strip(), texts, LETTERS.index(ans.group(1))


def mcq_question_agent(page, keywords, max_tries=3):

    prompt = f"""
You are a teaching assistant. Write ONE multiple-choice question about the lecture text below.

Rules:
- Use ONLY the lecture text.
- Exactly 4 options labeled A), B), C), D). Only one is correct.

Use exactly this format:
Question: <question>
A) <option>
B) <option>
C) <option>
D) <option>
Answer: <letter>

Lecture text:
{page.page_content}
"""
    for attempt in range(max_tries):

        parsed = parse_mcq(generate(prompt, max_new_tokens=200, sample=attempt > 0))

        if not parsed:
            continue

        question, options, answer_idx = parsed

        if support_score(options[answer_idx], page.page_content) < 0.6:
            continue

        return build_question("mcq", page_topic(page, keywords), page.metadata["page"], question, options,
                              answer_idx, f"From page {page.metadata['page']}: {page.page_content[:200]}")

    return None


def question_generator_agent(pages, keywords, n_llm=6, progress=None):

    questions = cloze_questions(pages, keywords)
    count = 0

    for p in pages:
        if count >= n_llm:
            break
        if len(p.page_content) < 150:
            continue
        if progress:
            progress((count, n_llm), desc=f"Generating question {count + 1}/{n_llm}")
        q = mcq_question_agent(p, keywords)
        if q:
            questions.append(q)
            count += 1

    for i, q in enumerate(questions):
        q["id"] = i

    return questions

## Answer Checker + Weak-Spot Tracker + Planner

In [14]:
page_text = lambda n: next(p.page_content for p in state["pages"] if p.metadata["page"] == n)


def answer_checker_agent(q, user_letter, explain=True):

    correct_letter = LETTERS[q["answer"]]

    if user_letter == correct_letter:
        return True, ""

    if not explain:
        return False, q["explanation"]

    prompt = f"""
A student answered a question wrongly.

Question: {q['question']}
Correct answer: {correct_letter}) {q['options'][q['answer']]}

In ONE or TWO short sentences, explain why the correct answer is right, using ONLY the lecture text below.

Lecture text:
{page_text(q['page'])}
"""
    feedback = clean_explanation(generate(prompt, max_new_tokens=100))

    return False, (q["explanation"] if feedback.startswith("Not covered") else feedback)


def topic_stats():

    if not state["attempts"]:
        return pd.DataFrame()

    df = (pd.DataFrame(state["attempts"]).groupby("topic")
          .agg(attempts=("correct", "size"), correct=("correct", "sum")).reset_index())

    df["accuracy"] = df["correct"] / df["attempts"]
    df["weakness"] = (df["attempts"] - df["correct"] + 1) / (df["attempts"] + 2)
    df["mastery"] = 1 - df["weakness"]

    return df


def weak_spot_tracker_agent():

    stats = topic_stats()

    if stats.empty:
        return []

    return stats[stats["weakness"] >= WEAK_THRESHOLD].sort_values("weakness", ascending=False)["topic"].tolist()


def planner_agent(pool, exam_date, max_days=7, per_day=3):

    stats = topic_stats()

    if stats.empty:
        return []

    days_left = max((exam_date - date.today()).days, 1)
    sessions = []

    for _, row in stats.sort_values("weakness", ascending=False).iterrows():
        sessions += [row["topic"]] * (2 if row["weakness"] >= WEAK_THRESHOLD else 1)

    n_days = min(days_left, max_days)
    per_day = max(per_day, ceil(len(sessions) / n_days))

    topic_pages = {}
    for q in pool:
        topic_pages.setdefault(q["topic"], set()).add(q["page"])

    plan = []

    for d in range(n_days):
        todays = list(dict.fromkeys(sessions[d * per_day:(d + 1) * per_day]))
        if not todays:
            break
        plan.append({"date": date.today() + timedelta(days=d + 1),
                     "tasks": [f"{t}: re-read the summary (pages {sorted(topic_pages.get(t, []))}) and redo its questions"
                               for t in todays]})

    if days_left >= 2:
        plan.append({"date": exam_date - timedelta(days=1),
                     "tasks": ["Full review: read all summaries and retake every question"]})

    return plan


def plan_to_text(plan):
    return "\n".join(f"{d['date']}:\n" + "\n".join(f"  - {t}" for t in d["tasks"]) for d in plan)

## Dashboard + PDF Report

In [15]:
def mastery_chart(stats):
    s = stats.sort_values("mastery")
    fig, ax = plt.subplots(figsize=(8, 0.6 * len(s) + 2))
    ax.barh(s["topic"], s["mastery"] * 100)
    ax.set_xlim(0, 100)
    ax.set_xlabel("Mastery (%)")
    ax.set_title("Topic mastery")
    return fig


def build_pdf_report(path):

    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.lib import colors
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image

    styles = getSampleStyleSheet()
    P = lambda t, s="BodyText": Paragraph(t, styles[s])

    def md_block(md_text):
        flow = []
        for line in md_text.splitlines():
            line = line.strip().replace("⚠", "(!)")
            if not line or line == "---":
                continue
            if line.startswith("##"):
                flow.append(P(escape(line.lstrip("# ").strip()), "Heading3")); continue
            m = re.match(r"\*\*(.+?)\*\*:\s*(.*)", line)
            if m:
                flow.append(P(f"<b>{escape(m.group(1))}</b>: {escape(m.group(2))}")); continue
            if line.startswith(("* ", "- ")):
                line = "• " + line[2:]
            flow.append(P(escape(line)))
        return flow

    flow = [P("Exam Prep Report", "Title"), Spacer(1, 12)]

    if state.get("summary_md"):
        flow += [P("1. Lecture Summary", "Heading2")] + md_block(state["summary_md"])

    if state.get("glossary_md"):
        flow += [P("2. Glossary", "Heading2")] + md_block(state["glossary_md"])

    if state.get("comparison_md"):
        rows = [[P(escape(c.strip())) for c in l.strip().strip("|").split("|")]
                for l in state["comparison_md"].splitlines()
                if l.startswith("|") and not set(l.replace("|", "").strip()) <= set("-: ")]
        t = Table(rows, repeatRows=1)
        t.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
                               ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey), ("VALIGN", (0, 0), (-1, -1), "TOP")]))
        flow += [P("3. Comparison", "Heading2"), t]

    if state.get("resources"):
        flow.append(P("4. Recommended Reading", "Heading2"))
        for term, papers in state["resources"].items():
            for p in papers:
                flow.append(P(f"<b>{escape(term)}</b>: {escape(p['title'])} ({escape(p['link'])})"))

    if state["wrong"]:
        flow.append(P("5. Questions You Missed", "Heading2"))
        for w in state["wrong"]:
            flow += [P(f"<b>{escape(w['topic'])}</b>: {escape(w['question'])}"),
                     P(f"Your answer: {escape(w['chosen'])} | Correct: {escape(w['correct'])}"),
                     P(f"Why: {escape(w['feedback'])}"), Spacer(1, 6)]

    stats = topic_stats()
    if not stats.empty:
        buf = io.BytesIO()
        mastery_chart(stats).savefig(buf, format="png", bbox_inches="tight")
        buf.seek(0)
        flow += [P("6. Mastery Dashboard", "Heading2"), Image(buf, width=400, height=220, kind="proportional")]

    if state["plan"]:
        flow.append(P("7. Revision Plan", "Heading2"))
        flow += [P(escape(l)) for l in plan_to_text(state["plan"]).splitlines()]

    SimpleDocTemplate(path, pagesize=A4).build(flow)

    return path

## Gradio Functions

In [16]:
def ui_load(file, progress=gr.Progress()):

    if file is None:
        return "⚠ Please upload a PDF or PPTX first.", ""

    try:
        progress(0.2, desc="Reading lecture...")
        pages, skipped = load_lecture(file)

        progress(0.6, desc="Building vector store...")
        state["pages"] = pages
        state["db"] = FAISS.from_documents(pages, embedding)

        for k in ("summary_md", "glossary_md", "comparison_md", "resources"):
            state.pop(k, None)
        reset_practice()

        keywords = get_keywords(pages, 8)

        return f"✅ {len(pages)} pages loaded (skipped: {skipped})", ", ".join(keywords)

    except Exception as e:
        return f"❌ Something went wrong: {e}", ""


def ui_summary(progress=gr.Progress()):

    if "pages" not in state:
        return "⚠ Load a lecture first.", None

    results = supervisor_agent(state["pages"], progress)
    state["summary_md"] = build_final_summary(results)
    Path("summary.md").write_text(state["summary_md"], encoding="utf-8")

    avg = sum(r["coverage"] for r in results) / len(results)

    return f"**Average coverage: {avg:.0%}**\n\n" + state["summary_md"], "summary.md"


def ui_glossary(keywords_text, progress=gr.Progress()):

    if "pages" not in state:
        return "⚠ Load a lecture first."

    state["glossary_md"] = build_glossary(parse_keywords(keywords_text), progress)

    return state["glossary_md"]


def ui_comparison(concepts_text, criteria_text):

    if "pages" not in state:
        return "⚠ Load a lecture first."

    concepts, criteria = parse_keywords(concepts_text), parse_keywords(criteria_text)

    if len(concepts) < 2 or not criteria:
        return "⚠ Enter at least 2 concepts and 1 criterion (comma separated)."

    state["comparison_md"] = comparison_agent(concepts, criteria)

    return state["comparison_md"]


def ui_resources(keywords_text, progress=gr.Progress()):

    terms = parse_keywords(keywords_text)
    found = {}

    for i, term in enumerate(terms):
        progress((i, len(terms)), desc=f"Searching arXiv: {term}")
        found[term] = resource_finder_agent(term)
        time.sleep(3)

    state["resources"] = found

    lines = [f"**{t}** — [{p['title']}]({p['link']})" for t, ps in found.items() for p in ps]

    return "\n\n".join(lines) or "No papers found."


def ui_make_pool(keywords_text, n_llm, progress=gr.Progress()):

    if "pages" not in state:
        return "⚠ Load a lecture first."

    reset_practice()
    state["pool"] = question_generator_agent(state["pages"], parse_keywords(keywords_text), int(n_llm), progress)

    return f"✅ {len(state['pool'])} questions ready. Press 'Start / Next round'."


def start_round():

    hidden = [gr.update(visible=False)] * MAX_BATCH

    if not state["pool"]:
        return ["⚠ Generate the question pool first."] + hidden + [""]

    if state["round"] >= MAX_ROUNDS:
        return ["🏁 Max rounds reached."] + hidden + [""]

    if state["round"] > 0 and not state["checked"]:
        return ["⚠ Submit your answers for this round first."] + [gr.update()] * MAX_BATCH + [""]

    r = state["round"] + 1
    pool = state["pool"]

    if r == 1:
        by_topic, batch, i = {}, [], 0
        for q in pool:
            by_topic.setdefault(q["topic"], []).append(q)
        while len(batch) < MAX_BATCH and any(i < len(v) for v in by_topic.values()):
            for v in by_topic.values():
                if i < len(v) and len(batch) < MAX_BATCH:
                    batch.append(v[i])
            i += 1
    else:
        weak = weak_spot_tracker_agent()
        if not weak:
            return ["🎉 No weak topics left!"] + hidden + [""]
        cand = [q for q in pool if q["topic"] in weak]
        fresh = [q for q in cand if q["id"] not in state["asked"]][:5]
        batch = fresh + [q for q in cand if q["id"] in state["asked"]][: 5 - len(fresh)]

    state.update(round=r, batch=batch, checked=False)

    radios = []
    for i in range(MAX_BATCH):
        if i < len(batch):
            q = batch[i]
            radios.append(gr.update(visible=True, value=None, label=f"Q{i + 1}. {q['question']}",
                                    choices=[f"{l}) {o}" for l, o in zip(LETTERS, q["options"])]))
        else:
            radios.append(gr.update(visible=False, value=None))

    return [f"### Round {r} / {MAX_ROUNDS} ({len(batch)} questions)"] + radios + [""]


def submit_round(explain, *answers):

    batch = state["batch"]

    if not batch:
        return "⚠ Start a round first."

    if state["checked"]:
        return "⚠ Already submitted. Start the next round."

    answers = answers[: len(batch)]

    if any(a is None for a in answers):
        return "⚠ Please answer all questions."

    lines = []

    for q, a in zip(batch, answers):

        letter = a[0]
        ok, feedback = answer_checker_agent(q, letter, explain)

        state["asked"].add(q["id"])
        state["attempts"].append({"topic": q["topic"], "correct": int(ok), "round": state["round"]})

        if ok:
            lines.append(f"✅ **{q['question']}**")
        else:
            lines.append(f"❌ **{q['question']}**\n\nCorrect: {LETTERS[q['answer']]}) {q['options'][q['answer']]}\n\nWhy: {feedback}")
            state["wrong"].append({"topic": q["topic"], "question": q["question"],
                                   "chosen": q["options"][LETTERS.index(letter)],
                                   "correct": q["options"][q["answer"]], "feedback": feedback})

    state["checked"] = True
    weak = weak_spot_tracker_agent()

    return "\n\n".join(lines) + f"\n\n---\n**Weak topics:** {', '.join(weak) if weak else 'none 🎉'}"


def ui_dashboard(exam_date_text):

    stats = topic_stats()

    if stats.empty:
        return None, None, "Finish at least one practice round first."

    try:
        exam_date = datetime.strptime(exam_date_text.strip(), "%Y-%m-%d").date()
    except ValueError:
        return None, None, "⚠ Exam date must look like 2026-12-10"

    state["plan"] = planner_agent(state["pool"], exam_date)

    show = stats.round(2)

    return mastery_chart(stats), show, plan_to_text(state["plan"])


def ui_pdf():
    return build_pdf_report("study_report.pdf")

## Gradio UI

In [17]:
default_exam = str(date.today() + timedelta(days=14))

with gr.Blocks() as demo:

    gr.Markdown("# 🎓 Exam Prep — Multi-Agent Assistant")

    with gr.Row():
        file_in = gr.File(label="Lecture (PDF / PPTX)", type="filepath")
        with gr.Column():
            load_btn = gr.Button("Load lecture", variant="primary")
            load_status = gr.Markdown()

    keywords_box = gr.Textbox(label="Keywords (comma separated)")

    load_btn.click(ui_load, inputs=file_in, outputs=[load_status, keywords_box])

    with gr.Tabs():

        with gr.Tab("📝 Summary"):
            sum_btn = gr.Button("Generate summary")
            sum_out = gr.Markdown()
            sum_file = gr.File(label="Download summary.md")
            sum_btn.click(ui_summary, outputs=[sum_out, sum_file])

        with gr.Tab("📖 Glossary"):
            gl_btn = gr.Button("Generate glossary")
            gl_out = gr.Markdown()
            gl_btn.click(ui_glossary, inputs=keywords_box, outputs=gl_out)

        with gr.Tab("⚖️ Comparison"):
            cmp_concepts = gr.Textbox(label="Concepts (comma separated)", value="Task Decomposition, Orchestration")
            cmp_criteria = gr.Textbox(label="Criteria (comma separated)", value="definition, purpose, example")
            cmp_btn = gr.Button("Compare")
            cmp_out = gr.Markdown()
            cmp_btn.click(ui_comparison, inputs=[cmp_concepts, cmp_criteria], outputs=cmp_out)

        with gr.Tab("🔎 Resources"):
            res_btn = gr.Button("Find papers on arXiv")
            res_out = gr.Markdown()
            res_btn.click(ui_resources, inputs=keywords_box, outputs=res_out)

        with gr.Tab("❓ Practice"):
            n_llm = gr.Slider(0, 12, value=6, step=1, label="LLM-generated questions")
            pool_btn = gr.Button("Generate question pool")
            pool_status = gr.Markdown()
            explain_chk = gr.Checkbox(label="Explain wrong answers with the LLM (slower)", value=False)
            round_btn = gr.Button("Start / Next round", variant="primary")
            round_md = gr.Markdown()
            radios = [gr.Radio(choices=[], visible=False, label="") for _ in range(MAX_BATCH)]
            submit_btn = gr.Button("Submit answers")
            fb_md = gr.Markdown()

            pool_btn.click(ui_make_pool, inputs=[keywords_box, n_llm], outputs=pool_status)
            round_btn.click(start_round, outputs=[round_md] + radios + [fb_md])
            submit_btn.click(submit_round, inputs=[explain_chk] + radios, outputs=fb_md)

        with gr.Tab("📅 Plan & Dashboard"):
            exam_box = gr.Textbox(label="Exam date (YYYY-MM-DD)", value=default_exam)
            dash_btn = gr.Button("Show dashboard + plan")
            dash_plot = gr.Plot()
            dash_df = gr.Dataframe()
            plan_out = gr.Textbox(label="Revision plan", lines=10)
            dash_btn.click(ui_dashboard, inputs=exam_box, outputs=[dash_plot, dash_df, plan_out])

        with gr.Tab("📄 Report"):
            pdf_btn = gr.Button("Build PDF report")
            pdf_file = gr.File(label="study_report.pdf")
            pdf_btn.click(ui_pdf, outputs=pdf_file)

## APP

In [ ]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d0f6c9253c3bbdb142.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
